<a href="https://colab.research.google.com/github/gns1719/Pet-NosePrint-Id-Service/blob/Jun/Siamese_test_0501.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics

In [ ]:
#코 비문 부분만 딴 거
!curl -L "https://app.roboflow.com/ds/2lyRjxG7V3?key=5t4vlqMXKC" > roboflow.zip; unzip roboflow.zip; rm roboflow.zip

In [ ]:
!cat /content/nose_data/data.yaml

In [ ]:
!pip install PyYAML

In [ ]:
import yaml

nose_data = {'train' : '/content/nose_data/train/images/',
        'test' : '/content/nose_data/test/images',
        'val' : '/content/nose_data/valid/images/',
        'names' : ['nose'],
        'nc': 1 }

# nose_data.yaml 파일 생성
with open('/content/nose_data/data.yaml', 'w') as f:
    yaml.dump(nose_data, f)

with open('/content/nose_data/data.yaml', 'r') as f:
    print(yaml.safe_load(f))

In [ ]:
from ultralytics import YOLO

# 모델 로드 (세그멘테이션 전용 사전학습 모델 사용)
model = YOLO('yolov8s-seg.pt')  # 또는 yolov8n-seg.pt, yolov8m-seg.pt 등

# 훈련
model.train(
    data='/content/nose_data/data.yaml',
    epochs=40,
    patience=10,
    imgsz=640,
    task='segment'  # 명시하지 않아도 자동 감지되지만, 명확히 넣어주는 게 좋음
)

# 추론
results = model.predict(
    source='/content/nose_test',
    save=True,  # 예측 이미지 저장
    save_crop=True,  # 감지된 세그멘테이션 객체 crop 저장
    project='/content/nose_dog',
    name='predict2',  # 프로젝트 내 폴더명
    task='segment'   # 세그멘테이션 명시
)

In [ ]:
import os
import cv2
import numpy as np
from pathlib import Path

# 원본 Siamese 폴더 구조가 있는 곳
input_dir = Path('/content/nose_test/siamese_dataset')
output_dir = Path('/content/nose_test/siamese_augmented_dataset')  # 증강 이미지 저장 위치
output_dir.mkdir(parents=True, exist_ok=True)

def augment_image(img):
    augmented = []

    # 원본 유지
    augmented.append(('original', img))

    # 흑백
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray_3ch = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    augmented.append(('gray', gray_3ch))

    # 좌우 반전
    flipped = cv2.flip(img, 1)
    augmented.append(('flipped', flipped))

    # 회전
    for angle in [-15, 15]:
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1)
        rotated = cv2.warpAffine(img, M, (w, h))
        augmented.append((f'rotated_{angle}', rotated))

    # 밝기 조절
    for factor in [40, -40]:
        bright = cv2.convertScaleAbs(img, alpha=1, beta=factor)
        augmented.append((f'bright_{factor:+}', bright))

    # 노이즈
    noise = np.random.normal(0, 15, img.shape).astype(np.uint8)
    noisy = cv2.add(img, noise)
    augmented.append(('noisy', noisy))

    return augmented

# 모든 dog 클래스 폴더에 대해 반복
for dog_dir in input_dir.glob('dog*'):
    if not dog_dir.is_dir():
        continue

    class_out_dir = output_dir / dog_dir.name
    class_out_dir.mkdir(parents=True, exist_ok=True)

    for img_path in dog_dir.glob('*.jpg'):
        img = cv2.imread(str(img_path))
        if img is None:
            continue

        base_name = Path(img_path).stem
        augmented_imgs = augment_image(img)

        for suffix, aug_img in augmented_imgs:
            new_filename = f'{base_name}_{suffix}.jpg'
            save_path = class_out_dir / new_filename
            cv2.imwrite(str(save_path), aug_img)

print("Siamese 구조에 맞춰 데이터 증강 완료!")

In [ ]:
import os
import csv
import random
from glob import glob

# 클래스 디렉토리 목록 가져오기
class_dirs = sorted([d for d in os.listdir(target_dir) if os.path.isdir(os.path.join(target_dir, d))])

# 클래스별 이미지 경로 저장
class_images = {}
for class_dir in class_dirs:
    class_path = os.path.join(target_dir, class_dir)
    images = sorted(glob(os.path.join(class_path, '*.jpg')))
    if len(images) >= 2:
        class_images[class_dir] = images

# Positive pairs 생성 (같은 클래스에서 1쌍씩)
positive_pairs = []
for class_dir, images in class_images.items():
    positive_pairs.append((images[0], images[1], 1))

# Negative pairs 생성 (다른 클래스 조합에서 무작위로 선택)
negative_pairs = []
class_list = list(class_images.keys())
while len(negative_pairs) < len(positive_pairs):
    class1, class2 = random.sample(class_list, 2)
    img1 = class_images[class1][0]
    img2 = class_images[class2][0]
    negative_pairs.append((img1, img2, 0))

# 모든 쌍을 합치고 셔플
all_pairs = positive_pairs + negative_pairs
random.shuffle(all_pairs)

# CSV 파일로 저장
csv_path = '/content/nose_test/siamese_pairs.csv'
with open(csv_path, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['image1', 'image2', 'label'])
    for pair in all_pairs:
        writer.writerow(pair)

print(f"총 {len(all_pairs)}개의 이미지 쌍이 {csv_path}에 저장되었습니다.")
print(f"Positive: {len(positive_pairs)}, Negative: {len(negative_pairs)}")


In [ ]:
!pip install torch torchvision

import os
import csv
import random
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

In [ ]:
# 하이퍼파라미터 설정
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 1e-4
IMG_SIZE = 224
CSV_PATH = '/content/nose_test/siamese_pairs.csv'

In [ ]:
class SiameseDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.pairs = []
        with open(csv_path, 'r') as f:
            reader = csv.reader(f)
            next(reader)  # 헤더 건너뛰기
            for row in reader:
                self.pairs.append(row)
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img1_path, img2_path, label = self.pairs[idx]
        img1 = Image.open(img1_path).convert('RGB')
        img2 = Image.open(img2_path).convert('RGB')
        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)
        return img1, img2, torch.tensor(float(label), dtype=torch.float32)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

dataset = SiameseDataset(CSV_PATH, transform=transform)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
class SiameseNetwork(nn.Module):
    def __init__(self):
        super(SiameseNetwork, self).__init__()
        self.backbone = models.resnet18(pretrained=True)
        self.backbone.fc = nn.Identity()  # 마지막 FC 레이어 제거
        self.fc = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward_once(self, x):
        return self.backbone(x)

    def forward(self, x1, x2):
        out1 = self.forward_once(x1)
        out2 = self.forward_once(x2)
        diff = torch.abs(out1 - out2)
        out = self.fc(diff)
        return torch.sigmoid(out)

In [ ]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, output, label):
        label = label.view(-1, 1)
        loss = label * torch.pow(output, 2) + \
               (1 - label) * torch.pow(torch.clamp(self.margin - output, min=0.0), 2)
        return torch.mean(loss)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SiameseNetwork().to(device)
criterion = ContrastiveLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for img1, img2, label in train_loader:
        img1, img2, label = img1.to(device), img2.to(device), label.to(device)
        optimizer.zero_grad()
        output = model(img1, img2)
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f'Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}')

torch.save(model.state_dict(), '/content/siamese_model.pth')
print("모델이 저장되었습니다.")

In [ ]:
!pip install torch torchvision pillow

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image

# SiameseNetwork 클래스에 ResNet18 또는 ResNet50을 선택할 수 있도록 수정
class SiameseNetwork(nn.Module):
    def __init__(self, backbone_type='resnet18'):
        super(SiameseNetwork, self).__init__()
        if backbone_type == 'resnet50':
            self.backbone = models.resnet50(pretrained=True)
            feature_dim = 2048
        else:
            self.backbone = models.resnet18(pretrained=True)
            feature_dim = 512

        self.backbone.fc = nn.Identity()  # 마지막 FC 레이어 제거

        self.fc = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward_once(self, x):
        return self.backbone(x)

    def forward(self, x1, x2):
        out1 = self.forward_once(x1)
        out2 = self.forward_once(x2)
        diff = torch.abs(out1 - out2)
        out = self.fc(diff)
        return torch.sigmoid(out)

# 디바이스 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ResNet50 사용 예시
model = SiameseNetwork(backbone_type='resnet18').to(device)

# 모델 가중치 로드 (resnet50으로 학습된 가중치를 사용해야 함)
model.load_state_dict(torch.load('/content/siamese_model.pth', map_location=device))
model.eval()

# 이미지 전처리 함수
def preprocess_image(image_path):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])
    image = Image.open(image_path).convert('RGB')
    return transform(image).unsqueeze(0)  # 배치 차원 추가

# 두 이미지 비교 함수
def compare_images(image_path1, image_path2):
    img1 = preprocess_image(image_path1).to(device)
    img2 = preprocess_image(image_path2).to(device)

    with torch.no_grad():
        embedding1 = model.forward_once(img1)
        embedding2 = model.forward_once(img2)
        similarity = F.cosine_similarity(embedding1, embedding2)
        similarity_score = similarity.item()

    print(f"코사인 유사도: {similarity_score:.4f}")
    if similarity_score > 0.7:
        print("두 이미지는 유사합니다.")
    else:
        print("두 이미지는 다릅니다.")

In [ ]:
# 비교할 이미지 경로 설정
image1_path = '/content/train2/crops/nose/coco2.jpg'
image2_path = '/content/train2/crops/nose/nono.jpg'

# 비교 실행
compare_images(image1_path, image2_path)